# Exercise 4

In [1]:
import math
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats

## Part 1

### Blocking simulation

In [2]:
def blocking(mean_interArrival,mean_service,customers,units):
    t_interArrival = np.random.exponential(mean_interArrival,customers)
    t_arrival = np.cumsum(t_interArrival)
    t_service = np.random.exponential(mean_service,customers)

    service_units = [0]*units
    n_blocked = 0
    for i in range(0,customers):
        t_departure = t_arrival[i]+t_service[i]
        
        vacancies = [x for x in service_units if x < t_arrival[i]]
        
        if vacancies:
            indx = service_units.index(vacancies[0])
            service_units[indx] = t_departure
            
        else:
            n_blocked += 1
      
    return n_blocked/customers

### Testing simulation

In [3]:
#Parameters
m = 10
mean_serviceT = 8
mean_interArrivalT = 1
n_repetitions = 10
n_customers = 10000

print("B_simulation:",blocking(mean_interArrivalT,mean_serviceT,n_customers,m))

B_simulation: 0.1267


### Comparing with theory

In [4]:
A = mean_interArrivalT * mean_serviceT

num = (A**m)/math.factorial(m)

denom = 0
for i in range(0,m):
    denom += ( (A**i)/math.factorial(i) )

B = num / denom

print("B_theory:",B)

B_theory: 0.13851266214160918


### Calculating confidence interval

In [5]:
reps_list = []
for i in range(n_repetitions):
    reps_list.append(blocking(mean_interArrivalT,mean_serviceT,n_customers,m))
reps = np.array(reps_list)


#Short method (same result):
conf_int = stats.t.interval(confidence = 0.95, df=n_repetitions-1, loc=np.mean(reps), scale=np.std(reps, ddof=1)/np.sqrt(n_repetitions))

print("CI:",conf_int)

#Long method (same result):
mean = np.mean(reps)
print("Mean:",mean)

#var = np.var(reps, ddof=1)

std = np.std(reps, ddof=1)

alpha=0.05
p_quantile_1 = stats.t.ppf(q=alpha/2,df=n_repetitions-1)
p_quantile_2 = stats.t.ppf(q=1-alpha/2,df=n_repetitions-1)

#conf_int = [mean + (std/np.sqrt(n_repetitions))*p_quantile_1,mean + (std/np.sqrt(n_repetitions))*p_quantile_2]

CI: (np.float64(0.11659257406844849), np.float64(0.12578742593155148))
Mean: 0.12118999999999999


## Part 2

In [6]:
def blocking_interArrival(t_interArrival,mean_service,customers,units):
    t_arrival = np.cumsum(t_interArrival)
    t_service = np.random.exponential(mean_service,customers)

    service_units = [0]*units
    n_blocked = 0
    for i in range(0,customers):
        t_departure = t_arrival[i]+t_service[i]
        
        vacancies = [x for x in service_units if x < t_arrival[i]]
        
        if vacancies:
            indx = service_units.index(vacancies[0])
            service_units[indx] = t_departure
            
        else:
            n_blocked += 1
      
    return n_blocked/customers

### a)

In [7]:
def Erlang(k,mean,customers):
    theta = mean/k

    return np.random.gamma(k,theta,customers)

In [8]:
k = 2
mean = 1
t_interArrival_Erlang = Erlang(k,mean,n_customers)

reps_list = []
for i in range(n_repetitions):
    reps_list.append(blocking_interArrival(t_interArrival_Erlang,mean_serviceT,n_customers,m))
reps = np.array(reps_list)

mean_Erlang = np.mean(reps)
print("Mean_Erlang:",mean_Erlang)

conf_int_Erlang = stats.t.interval(confidence = 0.95, df=n_repetitions-1, loc=np.mean(reps), scale=np.std(reps, ddof=1)/np.sqrt(n_repetitions))

print("CI_Erlang:",conf_int_Erlang)

Mean_Erlang: 0.09001
CI_Erlang: (np.float64(0.08679620607779008), np.float64(0.09322379392220993))


### b)

In [9]:
def hyperExp(p1,lambda1,p2,lambda2,n_customers):

    if p1+p2 == 1:
        choices = np.random.uniform(0,1,n_customers)

        first_choice = choices < p1

        t_interArrival = np.empty(n_customers)

        for i in range(n_customers):
            if first_choice[i] == True:
                t_interArrival[i] = np.random.exponential(1/lambda1)
            else:
                t_interArrival[i] = np.random.exponential(1/lambda2)

        return t_interArrival
    
    else:
        return "sum of probabilities is not 1"

In [ ]:
p1 = 0.8
lambda1 = 0.8333
p2 = 0.2
lambda2 = 5.0

t_interArrival_hyperExp = hyperExp(p1,lambda1,p2,lambda2,n_customers)

reps_list = []
for i in range(n_repetitions):
    reps_list.append(blocking_interArrival(t_interArrival_hyperExp,mean_serviceT,n_customers,m))
reps = np.array(reps_list)

mean_hyperExp = np.mean(reps)
print("Mean_hyperExp:",mean_hyperExp)

conf_int_hyperExp = stats.t.interval(confidence = 0.95, df=n_repetitions-1, loc=np.mean(reps), scale=np.std(reps, ddof=1)/np.sqrt(n_repetitions))

print("CI_hyperExp:",conf_int_hyperExp)

Mean_hyperExp: 0.13021000000000002
CI_hyperExp: (np.float64(0.126931397908127), np.float64(0.13348860209187305))


## Part 3

In [11]:
def blocking_service(mean_interArrival,t_service,customers,units):
    t_interArrival = np.random.exponential(mean_interArrival,customers)
    t_arrival = np.cumsum(t_interArrival)
    
    service_units = [0]*units
    n_blocked = 0
    for i in range(0,customers):
        t_departure = t_arrival[i]+t_service[i]
        
        vacancies = [x for x in service_units if x < t_arrival[i]]
        
        if vacancies:
            indx = service_units.index(vacancies[0])
            service_units[indx] = t_departure
            
        else:
            n_blocked += 1
      
    return n_blocked/customers

### a)

In [21]:
def constant(mean_service,customers):
    return np.full(customers,mean_service)

In [22]:
m = 10
mean_serviceT = 8
mean_interArrivalT = 1
n_repetitions = 10
n_customers = 10000

t_service_constant = constant(mean_serviceT,n_customers)

reps_list = []
for i in range(n_repetitions):
    reps_list.append(blocking_service(mean_interArrivalT,t_service_constant,n_customers,m))
reps = np.array(reps_list)

mean_constant = np.mean(reps)
print("Mean_constant:",mean_constant)

conf_int_constant = stats.t.interval(confidence = 0.95, df=n_repetitions-1, loc=np.mean(reps), scale=np.std(reps, ddof=1)/np.sqrt(n_repetitions))
print("CI_constant:",conf_int_constant)

Mean_constant: 0.12179000000000002
CI_constant: (np.float64(0.11999179452677113), np.float64(0.12358820547322892))


### b)

In [23]:
def pareto(k,customers):
    return np.random.pareto(k,customers)

In [26]:
m = 10
mean_serviceT = 8
mean_interArrivalT = 1
n_repetitions = 10
n_customers = 10000

k1 = 1.05
t_service_pareto1 = pareto(k1,n_customers)

blocking_service(mean_interArrivalT,t_service_pareto1,n_customers,m)

reps_list = []
for i in range(n_repetitions):
    reps_list.append(blocking_service(mean_interArrivalT,t_service_pareto1,n_customers,m))
reps = np.array(reps_list)

mean_pareto1 = np.mean(reps)
print("Mean_pareto1:",mean_pareto1)

conf_int_pareto1 = stats.t.interval(confidence = 0.95, df=n_repetitions-1, loc=np.mean(reps), scale=np.std(reps, ddof=1)/np.sqrt(n_repetitions))
print("CI_pareto1:",conf_int_pareto1)


k2 = 2.05
t_service_pareto2 = pareto(k2,n_customers)

blocking_service(mean_interArrivalT,t_service_pareto2,n_customers,m)

reps_list = []
for i in range(n_repetitions):
    reps_list.append(blocking_service(mean_interArrivalT,t_service_pareto2,n_customers,m))
reps = np.array(reps_list)

mean_pareto2 = np.mean(reps)
print("Mean_pareto2:",mean_pareto2)

conf_int_pareto2 = stats.t.interval(confidence = 0.95, df=n_repetitions-1, loc=np.mean(reps), scale=np.std(reps, ddof=1)/np.sqrt(n_repetitions))
print("CI_pareto2:",conf_int_pareto2)

Mean_pareto1: 0.08557000000000001
CI_pareto1: (np.float64(0.07936180375959202), np.float64(0.09177819624040799))
Mean_pareto2: 0.0
CI_pareto2: (np.float64(nan), np.float64(nan))


### c)

In [27]:
def normal(mean_service,std_service,customers):
    return np.random.normal(mean_service,std_service,customers)

In [28]:
m = 10
mean_serviceT = 8
mean_interArrivalT = 1
n_repetitions = 10
n_customers = 10000

std_mean = 4

t_service_normal = normal(mean_serviceT,std_mean,n_customers)

print("B_service_normal",blocking_service(mean_interArrivalT,t_service_normal,n_customers,m))


reps_list = []
for i in range(n_repetitions):
    reps_list.append(blocking_service(mean_interArrivalT,t_service_normal,n_customers,m))
reps = np.array(reps_list)

conf_int_normal = stats.t.interval(confidence = 0.95, df=n_repetitions-1, loc=np.mean(reps), scale=np.std(reps, ddof=1)/np.sqrt(n_repetitions))
print("CI_normal:",conf_int_normal)

B_service_normal 0.1171
CI_normal: (np.float64(0.12270033215529284), np.float64(0.12591966784470718))


## Part 4

In [29]:
#Part 1 (interArrival and service: exponenstial)
print("CI:",conf_int)

#Part 2 (interArrival: Erlang or hyperexponential)
print("CI_Erlang:",conf_int_Erlang)
print("CI_hyperExp:",conf_int_hyperExp)

#Part 3 (service: constant, pareto (two different k) or normal)
print("CI_constant:",conf_int_constant)
print("CI_pareto1:",conf_int_pareto1)
print("CI_pareto2:",conf_int_pareto2)
print("CI_normal:",conf_int_normal)

CI: (np.float64(0.11659257406844849), np.float64(0.12578742593155148))
CI_Erlang: (np.float64(0.08679620607779008), np.float64(0.09322379392220993))
CI_hyperExp: (np.float64(0.126931397908127), np.float64(0.13348860209187305))
CI_constant: (np.float64(0.11999179452677113), np.float64(0.12358820547322892))
CI_pareto1: (np.float64(0.07936180375959202), np.float64(0.09177819624040799))
CI_pareto2: (np.float64(nan), np.float64(nan))
CI_normal: (np.float64(0.12270033215529284), np.float64(0.12591966784470718))
